In [3]:
import pandas as pd
import numpy as np
import os

# 1. PENGATURAN PATH
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir)
file_path = os.path.join(base_dir, "DATASET", "rekap_harian_per_stasiun_komuter.xlsx")

print("Membaca file rekap per stasiun...")
xls = pd.ExcelFile(file_path)

sheet_bukan_stasiun = ['Dashboard', 'Panduan', 'Rekap_All', 'Per_Stasiun', 'Data_Asli']
daftar_stasiun = [sheet for sheet in xls.sheet_names if sheet not in sheet_bukan_stasiun]

# 2. SELEKSI 4 VARIABEL UTAMA YANG AKAN DIHASILKAN
kolom_pilihan = ['tanggal', 'nama_stasiun', 'penumpang_berangkat_komuter', 'penumpang_datang_komuter']

np.random.seed(42)
all_hourly_data = []

# 3. PROSES PERULANGAN UNTUK SETIAP STASIUN
for stasiun in daftar_stasiun:
    print(f"Memproses ekspansi 24 jam untuk Stasiun: {stasiun}")
    
    df_stasiun = pd.read_excel(file_path, sheet_name=stasiun, skiprows=1)
    df_stasiun.columns = ['tanggal', 'total_berangkat', 'total_datang', 'total_komuter']
    df_stasiun = df_stasiun.dropna(subset=['tanggal'])
    
    # Menentukan bobot distribusi waktu spesifik stasiun berdasarkan kondisi operasional terakhir
    # Kita set jam 02:00, 03:00, 04:00 mendekati atau sama dengan 0 karena istirahat operasional
    weights_berangkat = {
        0: 0.005, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.01, 5: 0.04,
        6: 0.15, 7: 0.18, 8: 0.12, 9: 0.05, 10: 0.03, 11: 0.03,
        12: 0.03, 13: 0.03, 14: 0.04, 15: 0.05, 16: 0.08, 17: 0.10,
        18: 0.05, 19: 0.02, 20: 0.01, 21: 0.005, 22: 0.005, 23: 0.001
    }

    weights_datang = {
        0: 0.01, 1: 0.005, 2: 0.0, 3: 0.0, 4: 0.005, 5: 0.01,
        6: 0.07, 7: 0.11, 8: 0.10, 9: 0.04, 10: 0.03, 11: 0.03,
        12: 0.03, 13: 0.03, 14: 0.04, 15: 0.06, 16: 0.14, 17: 0.16,
        18: 0.10, 19: 0.04, 20: 0.02, 21: 0.01, 22: 0.005, 23: 0.01
    }
    
    # LOGIKA KHUSUS KERETA TERAKHIR
    if stasiun == 'TANAHABANG':
        weights_berangkat[23] = 0.01  # Jam 23:00 keberangkatan terakhir masih tinggi di THB
        weights_berangkat[1] = 0.0   # Jam 01:00 sudah kosong total
        weights_datang[1] = 0.0
    elif stasiun == 'RANGKASBITUNG':
        weights_datang[1] = 0.015    # Jam 01:00 kedatangan penumpang terakhir di stasiun akhir
        weights_berangkat[1] = 0.0   # Jam 01:00 tidak ada yang berangkat lagi
        weights_berangkat[23] = 0.0
    else:
        # Untuk stasiun antara (antara THB dan RKB) jam 00:00 masih ada kereta lewat
        weights_datang[0] = 0.01
        weights_berangkat[1] = 0.0
        weights_datang[1] = 0.002    # Sisa-sisa kereta malam sebelum sampai RKB

    # Normalisasi bobot agar total presisi = 1.0
    sum_b = sum(weights_berangkat.values())
    sum_d = sum(weights_datang.values())
    w_b_norm = {k: v/sum_b for k, v in weights_berangkat.items()}
    w_d_norm = {k: v/sum_d for k, v in weights_datang.items()}

    for idx, row in df_stasiun.iterrows():
        tgl_berjalan = pd.to_datetime(row['tanggal']).strftime('%Y-%m-%d')
        
        for h in range(24):
            noise_b = np.random.uniform(0.85, 1.15)
            noise_d = np.random.uniform(0.85, 1.15)
            
            p_berangkat = np.round(row['total_berangkat'] * w_b_norm[h] * noise_b)
            p_datang = np.round(row['total_datang'] * w_d_norm[h] * noise_d)
            
            jam_str = f"{str(h).zfill(2)}:00"
            
            all_hourly_data.append({
                'tanggal': f"{tgl_berjalan} {jam_str}",
                'nama_stasiun': stasiun,
                'penumpang_berangkat_komuter': int(p_berangkat),
                'penumpang_datang_komuter': int(p_datang)
            })

# 4. MEMBUAT DATAFRAME DAN MENYIMPAN HASIL
df_final = pd.DataFrame(all_hourly_data)

# Memastikan urutan kolom sesuai dengan 4 variabel utama skripsi Anda
df_final = df_final[kolom_pilihan]

output_path = os.path.join(base_dir, "HASIL", "data_komuter_24jam_terstruktur.csv")
df_final.to_csv(output_path, index=False)

print("\n--- Hasil Pembuatan Data 24 Jam Sesuai Karakteristik Kereta Terakhir ---")
print(df_final.head(24))
print(f"\nSukses! Data seluruh stasiun kini sudah berbentuk per jam ({df_final.shape[0]} baris).")
print(f"File disimpan di folder HASIL dengan nama: \n{output_path}")

Membaca file rekap per stasiun...
Memproses ekspansi 24 jam untuk Stasiun: CICAYUR
Memproses ekspansi 24 jam untuk Stasiun: CIKOYA
Memproses ekspansi 24 jam untuk Stasiun: CILEJIT
Memproses ekspansi 24 jam untuk Stasiun: CISAUK
Memproses ekspansi 24 jam untuk Stasiun: CITERAS
Memproses ekspansi 24 jam untuk Stasiun: DARU
Memproses ekspansi 24 jam untuk Stasiun: JATAKE
Memproses ekspansi 24 jam untuk Stasiun: JURANGMANGU
Memproses ekspansi 24 jam untuk Stasiun: KEBAYORAN
Memproses ekspansi 24 jam untuk Stasiun: MAJA
Memproses ekspansi 24 jam untuk Stasiun: PALMERAH
Memproses ekspansi 24 jam untuk Stasiun: PARUNGPANJANG
Memproses ekspansi 24 jam untuk Stasiun: PONDOKRANJI
Memproses ekspansi 24 jam untuk Stasiun: RANGKASBITUNG
Memproses ekspansi 24 jam untuk Stasiun: RAWA BUNTU
Memproses ekspansi 24 jam untuk Stasiun: SERPONG
Memproses ekspansi 24 jam untuk Stasiun: SUDIMARA
Memproses ekspansi 24 jam untuk Stasiun: TANAHABANG
Memproses ekspansi 24 jam untuk Stasiun: TENJO
Memproses ekspan